# EDA — Vector detections (VectorNet observations)

Initial load of the local VectorNet observations occurrence table for exploratory analysis.

**Prerequisite:** run [`01_extract_vectors.ipynb`](01_extract_vectors.ipynb) first so `data/01_raw/vector/` contains the latest DwC-A extract.

This notebook starts with reading the dataset via the shared `load_vector_detections` helper. Full cleaning, plots, and feature work come in later cells. The observations archive is leaner than the legacy `vndatabase` extract (no published `countryCode`, `degreeOfEstablishment`, `higherGeography`, or `dynamicProperties` columns).

In [112]:
from __future__ import annotations

import json

from vector_guard.data import (
    default_vector_raw_dir,
    find_repo_root,
    load_vector_detections,
)

REPO_ROOT = find_repo_root()
RAW_DIR = default_vector_raw_dir(REPO_ROOT)
MANIFEST_PATH = RAW_DIR / "manifest.json"

df = load_vector_detections()
df.shape

(120158, 18)

## Source manifest

Metadata written by the extract notebook (fetch time, citation, checksum).

In [113]:
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"fetched_at: {manifest.get('fetched_at')}")
    print(f"source_url: {manifest.get('source_url')}")
    print(f"sha256:     {manifest.get('sha256')}")
    print(f"citation:   {manifest.get('citation')}")
else:
    print(f"No manifest at {MANIFEST_PATH}")
    manifest = None
manifest

fetched_at: 2026-11-11T09:08:50.835333+00:00
source_url: https://www.gbif.se/ipt/archive.do?r=vectornet_historical_data
sha256:     d42375422c9e63f392f55b80c1572632a7171fb5ea1b44326d47e2e03453ffa8
citation:   VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority (2025). VectorNet observations. Version 1.5. VectorNet. Occurrence dataset (CC-BY 4.0). https://doi.org/10.15468/f3k8r9


{'source_url': 'https://www.gbif.se/ipt/archive.do?r=vectornet_historical_data',
 'resource': 'vectornet_historical_data',
 'gbif_dataset_uuid': '4abd984b-122c-44a0-8c92-b37e2f5299b1',
 'fetched_at': '2026-11-11T09:08:50.835333+00:00',
 'zip_name': 'vectornet_dwca.zip',
 'zip_bytes': 6684735,
 'sha256': 'd42375422c9e63f392f55b80c1572632a7171fb5ea1b44326d47e2e03453ffa8',
 'citation': 'VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority (2025). VectorNet observations. Version 1.5. VectorNet. Occurrence dataset (CC-BY 4.0). https://doi.org/10.15468/f3k8r9',
 'extracted_files': ['.gitkeep', 'eml.xml', 'meta.xml', 'occurrence.txt']}

## First look

In [114]:
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}):")
print(df.dtypes)

Rows: 120,158
Columns (18):
id                        str
bibliographicCitation     str
basisOfRecord             str
occurrenceID              str
individualCount           str
lifeStage                 str
associatedTaxa            str
eventDate                 str
samplingProtocol          str
sampleSizeValue           str
sampleSizeUnit            str
higherGeographyID         str
locationAccordingTo       str
decimalLatitude           str
decimalLongitude          str
verbatimIdentification    str
identificationRemarks     str
scientificName            str
dtype: object


In [115]:
df.head().T

,0,1,2,3,4
id,1141be2c-6c03-4685-922b-d7e73d77bd0d,3768ff90-473f-450d-9af1-3ec808691f17,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,88d2c677-9895-4d5d-9311-ac6491c175e1,42e04dd7-fac1-43dd-a8dc-2460f669302a
bibliographicCitation,"Wilhelmsson et al,2021. Three Babesia species ...","Wilhelmsson et al,2021. Three Babesia species ...","Wilhelmsson et al,2021. Three Babesia species ...","Borsan et al,2021. High Diversity, Prevalence,...","Vikentjeva et al.,2021. Rickettsia spp. in rod..."
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
occurrenceID,1141be2c-6c03-4685-922b-d7e73d77bd0d,3768ff90-473f-450d-9af1-3ec808691f17,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,88d2c677-9895-4d5d-9311-ac6491c175e1,42e04dd7-fac1-43dd-a8dc-2460f669302a
individualCount,0,0,4,4,75
lifeStage,Larvae,Adults,Nymph/Pupa,Larvae,NaN
associatedTaxa,Acrocephalus palustris,Acrocephalus palustris,Acrocephalus palustris,Apodemus agrarius,Apodemus flavicollis
eventDate,2009-03-15/2009-11-15,2009-03-15/2009-11-15,2009-03-15/2009-11-15,2018-03-01/2018-11-30,2012-04-01/2014-11-30
samplingProtocol,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts
sampleSizeValue,21,21,21,NaN,NaN


In [116]:
# Null summary with % of nulls over total rows
import pandas as pd
def calculate_null_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.isnull()
        .sum()
        .to_frame("null_count")
        .assign(
            null_pct=lambda x: (x["null_count"] / len(df) * 100).round(2),
            nunique=df.nunique(),
            dtype=df.dtypes,
            sample_values=df.apply(
                lambda s: s.dropna().unique()[:5].tolist()
            ),
        )
        .sort_values("null_count", ascending=False)
    )
calculate_null_summary(df)

,null_count,null_pct,nunique,dtype,sample_values
associatedTaxa,67467,56.15,56,str,"[Acrocephalus palustris, Apodemus agrarius, Ap..."
sampleSizeValue,30149,25.09,194,str,"[21, 30, 15, 2, 1]"
sampleSizeUnit,6017,5.01,15,str,"[hosts, trans, hours, trnig, m3h]"
lifeStage,1364,1.14,4,str,"[Larvae, Adults, Nymph/Pupa, Eggs]"
bibliographicCitation,0,0.00,535,str,"[Wilhelmsson et al,2021. Three Babesia species..."
id,0,0.00,120158,str,"[1141be2c-6c03-4685-922b-d7e73d77bd0d, 3768ff9..."
individualCount,0,0.00,1804,str,"[0, 4, 75, 5, 83]"
occurrenceID,0,0.00,120158,str,"[1141be2c-6c03-4685-922b-d7e73d77bd0d, 3768ff9..."
eventDate,0,0.00,9399,str,"[2009-03-15/2009-11-15, 2018-03-01/2018-11-30,..."
basisOfRecord,0,0.00,1,str,[humanObservation]


# Derive countryCode from higherGeographyID

The observations DwC-A has no `countryCode` column. Use the first two characters of `higherGeographyID` (NUTS/GAUL-style codes) as a working ISO-2 proxy.

In [117]:
# observations archive: derive countryCode (no published countryCode column)
df["countryCode"] = (
    df["higherGeographyID"].astype("string").str[:2].str.upper()
)
df["countryCode"].isnull().sum()

np.int64(0)

In [118]:
# sanity check: derived countryCode should match higherGeographyID prefix
def country_code_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    prefix = df["higherGeographyID"].astype("string").str[:2].str.upper()
    return df[df["countryCode"].ne(prefix) & df["higherGeographyID"].notna()]

mismatches = country_code_mismatches(df)
mismatches

,id,bibliographicCitation,basisOfRecord,occurrenceID,individualCount,lifeStage,associatedTaxa,eventDate,samplingProtocol,sampleSizeValue,sampleSizeUnit,higherGeographyID,locationAccordingTo,decimalLatitude,decimalLongitude,verbatimIdentification,identificationRemarks,scientificName,countryCode


# Split eventDate in start / end interval

In [119]:
# split eventDate in start / end interval date columns
parts = df["eventDate"].str.split("/", n=1, expand=True)
df["eventDate_start"] = pd.to_datetime(parts[0], errors="coerce")
df["eventDate_end"] = pd.to_datetime(parts[1], errors="coerce")

## Normalize eventDate interval

In [120]:
s, e = df["eventDate_start"], df["eventDate_end"]

both_na   = s.isna() & e.isna()   # NA/NA
start_only = s.isna() & e.notna() # NA/correct-date
end_only   = s.notna() & e.isna() # correct-date/NA

# collapse one-sided intervals to a point
df.loc[start_only, "eventDate_start"] = df.loc[start_only, "eventDate_end"]
df.loc[end_only, "eventDate_end"] = df.loc[end_only, "eventDate_start"]

df["eventDate_missing"] = both_na  # drop/filter later for modelling panel

In [121]:
# flag inverted intervals
inverted = df["eventDate_end"] < df["eventDate_start"]
df["eventDate_originally_inverted"] = inverted


inverted_set = df[df["eventDate_originally_inverted"] == True]
print(f"Number of inverted intervals: {len(inverted_set)}")
inverted_set.head().T

Number of inverted intervals: 6584


,563,572,587,683,685
id,90c85220-c71e-4fc6-9fe5-c1e93e0f5db5,b24c6e90-63ee-4b05-a397-75f7ea0f6f02,4d84de4c-551c-4057-a457-8feac8e1d2c8,e54fb8ab-16db-4efc-87f2-37e5e15b90e7,6a5aaf06-7aa6-48d2-8364-c5c8405ba10f
bibliographicCitation,"Rozek W, Larska M, Kwasnik M,2021. National Cu...","Sanders CJ, Shortall CR, England M, Harrington...","Sanders CJ, Shortall CR, England M, Harrington...","Sanders CJ, Shortall CR, England M, Harrington...","Sanders CJ, Shortall CR, England M, Harrington..."
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
occurrenceID,90c85220-c71e-4fc6-9fe5-c1e93e0f5db5,b24c6e90-63ee-4b05-a397-75f7ea0f6f02,4d84de4c-551c-4057-a457-8feac8e1d2c8,e54fb8ab-16db-4efc-87f2-37e5e15b90e7,6a5aaf06-7aa6-48d2-8364-c5c8405ba10f
individualCount,4,0,0,0,0
lifeStage,Adults,Adults,Adults,Adults,Adults
associatedTaxa,Not applicable,Not applicable,Not applicable,Not applicable,Not applicable
eventDate,2022-07-08/2020-01-09,1980-06-08/1978-03-09,2014-02-05/2012-03-06,2013-02-08/2012-10-08,2012-03-09/2011-11-09
samplingProtocol,Onderstepoort Veterinary Institute (OVI) black...,Rothamsted trap,Rothamsted trap,Rothamsted trap,Rothamsted trap
sampleSizeValue,1,2700,2700,2700,2700


In [122]:
# canonicalize: earlier → start, later → end
bounds = df[["eventDate_start", "eventDate_end"]]
df["eventDate_start"] = bounds.min(axis=1)
df["eventDate_end"] = bounds.max(axis=1)

still_inverted = df[df["eventDate_end"] < df["eventDate_start"]]
print(f"Number of inverted intervals: {len(still_inverted)}")


Number of inverted intervals: 0


In [132]:
# detect parse failures / remaining odd formats
still_to_fix = df[~df["eventDate_missing"] & (df["eventDate_start"].isna() | df["eventDate_end"].isna())]
print(f"Number of still to fix: {len(still_to_fix)}")

Number of still to fix: 0


In [123]:
missing_dates = df[df["eventDate_missing"] == True]
print(f"Number of missing dates: {len(missing_dates)}")
missing_dates.head().T

Number of missing dates: 993


,193,200,210,214,216
id,2555204f-263a-43f8-a45e-ef245b57d8ef,cb324e1a-40b2-4a31-b6d4-af30e22e0da4,0c33e1e2-2c4e-43c0-870f-bb894c763971,2c5827b4-06f8-425b-a52d-0cba3e74cdc1,1f679e3b-428e-499d-90c1-563b386ed3d7
bibliographicCitation,"Geurden et al,2018. Detection of tick-borne pa...","Geurden et al,2018. Detection of tick-borne pa...","Geurden et al,2018. Detection of tick-borne pa...","Geurden et al,2018. Detection of tick-borne pa...","Geurden et al,2018. Detection of tick-borne pa..."
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
occurrenceID,2555204f-263a-43f8-a45e-ef245b57d8ef,cb324e1a-40b2-4a31-b6d4-af30e22e0da4,0c33e1e2-2c4e-43c0-870f-bb894c763971,2c5827b4-06f8-425b-a52d-0cba3e74cdc1,1f679e3b-428e-499d-90c1-563b386ed3d7
individualCount,8,7,1,6,6
lifeStage,NaN,NaN,NaN,NaN,NaN
associatedTaxa,Canis lupus familiaris,Canis lupus familiaris,Canis lupus familiaris,Canis lupus familiaris,Canis lupus familiaris
eventDate,NA/NA,NA/NA,NA/NA,NA/NA,NA/NA
samplingProtocol,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts
sampleSizeValue,NaN,NaN,NaN,NaN,NaN


# Final resume

In [134]:
null_summary = calculate_null_summary(df)
null_summary[null_summary["null_count"] < 1.1 * len(df)]

,null_count,null_pct,nunique,dtype,sample_values
associatedTaxa,67467,56.15,56,str,"[Acrocephalus palustris, Apodemus agrarius, Ap..."
sampleSizeValue,30149,25.09,194,str,"[21, 30, 15, 2, 1]"
sampleSizeUnit,6017,5.01,15,str,"[hosts, trans, hours, trnig, m3h]"
lifeStage,1364,1.14,4,str,"[Larvae, Adults, Nymph/Pupa, Eggs]"
eventDate_start,993,0.83,5403,datetime64[us],"[2009-03-15 00:00:00, 2018-03-01 00:00:00, 201..."
eventDate_end,993,0.83,5483,datetime64[us],"[2009-11-15 00:00:00, 2018-11-30 00:00:00, 201..."
id,0,0.00,120158,str,"[1141be2c-6c03-4685-922b-d7e73d77bd0d, 3768ff9..."
occurrenceID,0,0.00,120158,str,"[1141be2c-6c03-4685-922b-d7e73d77bd0d, 3768ff9..."
individualCount,0,0.00,1804,str,"[0, 4, 75, 5, 83]"
bibliographicCitation,0,0.00,535,str,"[Wilhelmsson et al,2021. Three Babesia species..."


In [126]:
def print_col_info(col):
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print(df[col].value_counts().head())
    print()


print_col_info("eventDate")

eventDate
<StringArray>
['2009-03-15/2009-11-15', '2018-03-01/2018-11-30', '2012-04-01/2014-11-30',
 '2013-04-01/2014-11-30', '2012-04-01/2012-11-30']
Length: 5, dtype: str
9399
eventDate
2020-01-01/2025-07-08    12415
2018-06-01/2018-09-30     9618
2020-01-01/2023-08-31     3000
2009-01-01/2009-12-31     1651
NA/NA                      993
Name: count, dtype: int64

